# Belimo GL code: rules test

**Question:** if we hide the GL code (column M), how well can rules guess it?

| Step | What happens |
|---|---|
| 1 | Load the **Downloads Completed file** (real UPS export, GL known). This is the **test** set. |
| 2 | Load **Belimo UPS rows from the Supabase DB** and rebuild UPS-style columns from them. This is the **training** set. |
| 3 | Score the **doc rules** (`GL Code Logic.docx`, stored in `rules/gl_rules.json`) on the test set. |
| 4 | Train a **decision tree** on the DB rows, tune it on the newest DB rows, score it once on the test set. |
| 5 | Compare, and look at what both get wrong. |

Run from inside `gl_code/`. It needs `SUPABASE_DB_URL` in the repo-root `.env` and the Completed Belimo csv in your Downloads folder.

In [1]:
import json, re, itertools, os
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier, export_text

pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60); pd.set_option("display.max_columns", 30)

TEST_FILE = Path.home() / "Downloads" / "UPS GL Code Logic Completed_Belimo_100226.csv"
RULES_FILE = Path("rules/gl_rules.json")
DEFAULT_GL = "353500"            # most common Belimo code: what an unmatched row falls back to
DB_COLS = ["C", "P", "Q", "BP", "BS", "BX", "CA", "CD"]   # UPS columns the DB can give us
load_dotenv(Path("..") / ".env")

True

In [2]:
def col_idx(letter):                      # 'M' -> 12, 'AQ' -> 42
    n = 0
    for ch in letter.upper(): n = n * 26 + ord(ch) - 64
    return n - 1

def norm_gl(g):                           # long dash -> '--'
    return re.sub(r"[\u2014\u2015]", "--", str(g).strip())

def read_ups(path):                       # UPS files have no header row
    for enc in ["utf-8", "latin1"]:
        try: return pd.read_csv(path, header=None, dtype=str, keep_default_na=False, encoding=enc)
        except UnicodeDecodeError: pass

def lower_cols(raw):                      # the columns we use, lowercased text
    return {c: raw[col_idx(c)].fillna("").astype(str).str.lower() for c in DB_COLS}

## 1. Test set: the Downloads Completed file

In [3]:
test = read_ups(TEST_FILE)
y_test = test[col_idx("M")].map(norm_gl).values
print(test.shape, "(rows, columns)  - dates:", test[col_idx("L")].min(), "to", test[col_idx("L")].max())
dist = pd.Series(y_test).value_counts().rename("rows").to_frame()
dist["share %"] = (100 * dist.rows / len(y_test)).round(1)
dist

(8705, 246) (rows, columns)  - dates: 16-Sep-26 to 9/9/2026


,rows,share %
353500,7906,90.8
356200,258,3.0
367211,145,1.7
369600,142,1.6
353590,141,1.6
356205,70,0.8
367400,17,0.2
367210,13,0.1
353550,7,0.1
367230,6,0.1


`353500` is about 91% of rows, so **always guessing `353500` already scores ~91%**. The number that matters is how well we do on the other ~9% (the *non-default* rows).

## 2. Training set: Belimo UPS rows from the DB
The DB doesn't store the raw UPS file, only a processed copy, so the UPS columns are rebuilt from these fields:

In [4]:
MAPPING = pd.DataFrame([
    ["C",  "account", "account_no"], ["P", "reference 1", "dist_code"], ["Q", "reference 2", "status_info"],
    ["BP", "sender", "company_from"], ["BS", "sender city", "city_from"],
    ["BX", "receiver", "company_to"], ["CA", "receiver city", "city_to"], ["CD", "receiver country", "country_abbreviation_to"],
    ["AJ / AM / BW", "txn type / user id / receiver attention", "NOT IN DB (blank)"],
], columns=["UPS col", "meaning", "DB column"])
MAPPING

,UPS col,meaning,DB column
0,C,account,account_no
1,P,reference 1,dist_code
2,Q,reference 2,status_info
3,BP,sender,company_from
4,BS,sender city,city_from
5,BX,receiver,company_to
6,CA,receiver city,city_to
7,CD,receiver country,country_abbreviation_to
8,AJ / AM / BW,txn type / user id / receiver attention,NOT IN DB (blank)


In [5]:
engine = create_engine(os.environ["SUPABASE_DB_URL"].replace("postgresql://", "postgresql+psycopg2://", 1).replace("postgres://", "postgresql+psycopg2://", 1))
with engine.connect() as c:
    db = pd.read_sql(text("""select * from tdx_src.tde_tracking
                          where client_name ilike '%belimo%' and carrier_name = 'UPS'
                          order by sf_created_date, id"""), c).fillna("").astype(str).replace({"None": "", "NaT": ""})

def to_ups_columns(u):
    raw = pd.DataFrame("", index=range(len(u)), columns=range(246))
    for L, s in {"C": u.account_no, "P": u.dist_code, "Q": u.status_info, "BP": u.company_from, "BS": u.city_from,
                 "BX": u.company_to, "CA": u.city_to, "CD": u.country_abbreviation_to}.items():
        raw[col_idx(L)] = s.values
    return raw

db_raw = to_ups_columns(db)
y_db = db.gl_code.map(norm_gl).values
print(len(db), "DB rows, created", db.sf_created_date.iloc[0][:10], "->", db.sf_created_date.iloc[-1][:10])
pd.Series(y_db).value_counts().head(8).rename("rows").to_frame()

23929 DB rows, created 2026-01-07 -> 2026-02-27


,rows
353500,21277
356200,969
367230,524
369600,385
353590,309
367211,195
353550,140
356205,92


## 3. The doc rules
Loaded from `rules/gl_rules.json` (parsed from `GL Code Logic.docx`, with three typos fixed). A rule fires when **all** its clauses hold; a clause holds when **any** of its alternatives holds. If several rules fire, the **first one in doc order** wins (the doc doesn't say which should). Rows no rule fires on get the default `353500`.

In [6]:
def cond_mask(cols, op, val):             # cols = dict of lowercased columns
    pass

def apply_rules(raw, rules, default=None):
    n = len(raw); S = {}
    def col(c):
        if c not in S: S[c] = raw[col_idx(c)].fillna("").astype(str).str.lower()
        return S[c]
    def test(c, op, v):
        s, v = col(c), str(v).lower()
        if op == "eq":           return (s == v.strip()).values
        if op == "contains":     return s.str.contains(re.escape(v), regex=True).values
        if op == "not_contains": return (~s.str.contains(re.escape(v), regex=True)).values
        pat = r"(?<![a-z0-9\-\.])" + re.escape(v) + r"(?![a-z0-9\-\.])"
        m = s.str.contains(pat, regex=True).values
        return m if op == "word" else ~m
    hits = []
    for r in rules:
        m = np.ones(n, bool)
        for alts in r["when"]:
            a = np.zeros(n, bool)
            for c, op, v in alts: a |= test(c, op, v)
            m &= a
        hits.append(m & (r["gl"] is not None))
    H = np.column_stack(hits); gl = np.array([r["gl"] for r in rules], dtype=object)
    fired = H.any(axis=1); first = H.argmax(axis=1)
    pred = np.where(fired, gl[first], default)
    return pred, fired, np.where(fired, first, -1)

RULES = json.loads(RULES_FILE.read_text())["rule_sets"]
doc_rules = RULES["doc_v1"]["rules"]["Belimo"]
print(len(doc_rules), "doc rules for Belimo")
pd.DataFrame([{"id": r["id"], "GL": r["gl"], "doc line / note": (r.get("doc_line") or r.get("note") or "")[:110]} for r in doc_rules]).head(25)

23 doc rules for Belimo


,id,GL,doc line / note
0,D-B01,356200,"If Column C contains 9V1790, column M = 356200"
1,D-B02,367211,"If Column P contains sample, column M = 367211"
2,D-B23,367210,added: Trade + Promo together = 367210 (not in doc; must...
3,D-B03,367400,"If Column P contains Trade, column M = 367400"
4,D-B04,367230,"If Column P contains Lit, column M = 367230"
5,D-B05,367230,"If Column P contains Promo, column M = 367230"
6,D-B06,353590,"If Column P contains RMA, column M = 353590"
7,D-B07,353590,"If Column P contains 2-10, column M = 367230"
8,D-B08,369600,If Column BP contains Imprint and Column BS contains Osh...
9,D-B09,369600,If Column BY contains Imprint and Column CA contains Osh...


In [7]:
doc_pred, doc_fired, doc_rule = apply_rules(test, doc_rules, default=DEFAULT_GL)
print("rows where a doc rule fired:", doc_fired.sum(), f"({100*doc_fired.mean():.1f}%)")

rows where a doc rule fired: 953 (10.9%)


## 4. A tree trained on the DB rows
Each text column is split into words (`BX:belimo`, `P:rma`, ...). A decision tree learns which words decide the GL.

* **Train:** the oldest 70% of DB rows (by created date)
* **Validation:** the newest 30%, used only to pick the tree settings
* **Test:** the Downloads file, scored once at the end

In [8]:
class Words:
    def __init__(self, min_df): self.min_df = min_df
    def fit(self, D):
        self.v, M = {}, []
        for c in DB_COLS:
            v = CountVectorizer(binary=True, token_pattern=r"[a-z0-9][a-z0-9\-\.]*", min_df=self.min_df)
            try: M.append(v.fit_transform(D[c])); self.v[c] = v
            except ValueError: pass
        self.names = [f"{c}:{w}" for c in self.v for w in self.v[c].get_feature_names_out()]
        return sp.hstack(M).tocsr()
    def transform(self, D): return sp.hstack([self.v[c].transform(D[c]) for c in self.v]).tocsr()

D_db = lower_cols(db_raw)
cut = int(len(db) * 0.7); tr, va = np.arange(cut), np.arange(cut, len(db))
rows = []
for depth, leaf, mdf in itertools.product([4, 6, 8, 12], [2, 5, 20], [2, 5, 10]):
    w = Words(mdf); X = w.fit({c: D_db[c].iloc[tr] for c in DB_COLS})
    t = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=0).fit(X, y_db[tr])
    p = t.predict(w.transform({c: D_db[c].iloc[va] for c in DB_COLS}))
    rows.append(dict(depth=depth, min_leaf=leaf, min_df=mdf, leaves=t.get_n_leaves(), val_accuracy=round(100 * (p == y_db[va]).mean(), 2)))
grid = pd.DataFrame(rows).sort_values(["val_accuracy", "leaves"], ascending=[False, True])
print(f"validation: always-{DEFAULT_GL} = {100*(y_db[va]==DEFAULT_GL).mean():.2f}%")
grid.head(6)

validation: always-353500 = 90.39%


,depth,min_leaf,min_df,leaves,val_accuracy
28,12,2,5,48,97.06
30,12,5,2,47,97.03
31,12,5,5,47,97.03
29,12,2,10,56,97.03
32,12,5,10,45,96.98
27,12,2,2,49,96.95


In [9]:
best = grid.iloc[0]
words = Words(int(best.min_df)); X = words.fit(D_db)               # retrain on ALL DB rows
tree = DecisionTreeClassifier(max_depth=int(best.depth), min_samples_leaf=int(best.min_leaf), random_state=0).fit(X, y_db)
print(f"final tree: depth {tree.get_depth()}, {tree.get_n_leaves()} leaves, trained on {len(db)} DB rows")
print(export_text(tree, feature_names=words.names, max_depth=4)[:2500])

final tree: depth 12, 56 leaves, trained on 23929 DB rows
|--- BX:belimo <= 0.50
|   |--- P:lit <= 0.50
|   |   |--- P:sample <= 0.50
|   |   |   |--- P:rma <= 0.50
|   |   |   |   |--- CD:us <= 0.50
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |   |--- CD:us >  0.50
|   |   |   |   |   |--- truncated branch of depth 8
|   |   |   |--- P:rma >  0.50
|   |   |   |   |--- class: 353590
|   |   |--- P:sample >  0.50
|   |   |   |--- P:12 <= 0.50
|   |   |   |   |--- class: 367211
|   |   |   |--- P:12 >  0.50
|   |   |   |   |--- class: 353550
|   |--- P:lit >  0.50
|   |   |--- class: 367230
|--- BX:belimo >  0.50
|   |--- BS:oshkosh <= 0.50
|   |   |--- BP:msc <= 0.50
|   |   |   |--- BP:belimo <= 0.50
|   |   |   |   |--- Q:for <= 0.50
|   |   |   |   |   |--- truncated branch of depth 8
|   |   |   |   |--- Q:for >  0.50
|   |   |   |   |   |--- class: 369600
|   |   |   |--- BP:belimo >  0.50
|   |   |   |   |--- CD:ca <= 0.50
|   |   |   |   |   |--- truncated br

## 5. Results on the test file (Downloads, Jul–Sep 2026)

In [10]:
tree_pred = tree.predict(words.transform(lower_cols(test)))
base_pred = np.full(len(y_test), DEFAULT_GL, dtype=object)
nd = y_test != DEFAULT_GL

def score(p): return {"overall %": round(100*(p == y_test).mean(), 2),
                      f"non-default rows ({nd.sum()}) %": round(100*(p[nd] == y_test[nd]).mean(), 2),
                      f"default rows ({(~nd).sum()}) %": round(100*(p[~nd] == y_test[~nd]).mean(), 2)}
summary = pd.DataFrame({"always 353500": score(base_pred), "doc rules (+default)": score(doc_pred), "tree trained on DB": score(tree_pred)}).T
summary

,overall %,non-default rows (799) %,default rows (7906) %
always 353500,90.82,0.00,100.00
doc rules (+default),97.59,82.60,99.10
tree trained on DB,96.07,68.59,98.85


In [11]:
per = pd.DataFrame({"rows": pd.Series(y_test).value_counts()})
for name, p in [("doc %", doc_pred), ("tree %", tree_pred)]:
    per[name] = [round(100 * (p[y_test == g] == g).mean(), 1) for g in per.index]
per.index.name = "real GL"
per

,rows,doc %,tree %
real GL,,,
353500,7906,99.1,98.8
356200,258,100.0,94.6
367211,145,100.0,97.9
369600,142,46.5,46.5
353590,141,100.0,31.2
356205,70,28.6,35.7
367400,17,100.0,82.4
367210,13,100.0,0.0
353550,7,0.0,100.0


## 6. What both get wrong
The biggest shared misses are `353590` (RMA) and `369600` (marketing). Here is what those rows look like in the test file.

In [12]:
wrong = pd.DataFrame({"real": y_test, "doc": doc_pred, "tree": tree_pred})
print("most common mistakes (real -> predicted by doc):")
wrong[wrong.real != wrong.doc].groupby(["real", "doc"]).size().sort_values(ascending=False).head(8).rename("rows").to_frame()

most common mistakes (real -> predicted by doc):


,,rows
real,doc,
369600,356200,68
353500,356200,51
356205,353500,36
353500,356205,18
356205,356200,14
369600,353500,8
353550,353500,7
367230,367210,3


In [13]:
SHOW = {"C": "acct", "P": "ref1", "Q": "ref2", "AJ": "txn", "AM": "user", "BP": "sender", "BW": "recv attn", "BX": "receiver"}
def show(real_gl, n=12):
    m = (y_test == real_gl) & (doc_pred != real_gl)
    out = pd.DataFrame({name: test[col_idx(L)] for L, name in SHOW.items()})[m]
    out.insert(0, "doc said", doc_pred[m])
    print(f"{real_gl}: {m.sum()} rows the doc rules got wrong")
    return out.head(n)
show("353590")

353590: 0 rows the doc rules got wrong


,doc said,acct,ref1,ref2,txn,user,sender,recv attn,receiver


In [14]:
show("369600")

369600: 76 rows the doc rules got wrong


,doc said,acct,ref1,ref2,txn,user,sender,recv attn,receiver
347,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
348,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
349,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
350,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
351,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
352,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
353,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
354,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
361,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."
362,356200,97644,No Reference,,MAN,,"BELIMO US, Inc.",Marissa Palmer,"BELIMO US, Inc."


## Notes and limits

* The DB has no **AJ** (transaction type), **AM** (user id) or **BW** (receiver attention). Rules that need them can't fire on DB rows, and the tree trained on the DB can't use them. The test file has all columns, so the doc rules could in principle use BW, but they don't for Belimo.
* DB rows are one row per **shipment**; the test file is one row per **charge line**. Accuracy is per row, so shipments with many lines weigh more in the test.
* The doc has no priority between rules, so "first rule in doc order" is an assumption. See `rules/gl_rules.json` for the exact rules.
* The test file is the data the v2 notebook rules were developed from. That doesn't affect these results, because neither the doc rules (v1) nor the DB-trained tree used it.